# Q1: Image Denoising with a Convolutional Denoising Autoencoder (DAE)
Dataset: Berkeley Segmentation Dataset (BSDS300). Framework: PyTorch. Runs on Google Colab (GPU recommended).

Sections: 1. Data  2. Noise injection  3. Model  4. Training  5. Evaluation and visualisation  6. Experimental study

In [ ]:
!pip -q install scikit-image
import os, tarfile, urllib.request, glob, random
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from skimage.metrics import structural_similarity as ssim_fn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 128
print("Device:", DEVICE)

## 1. Dataset preparation
Download BSDS300 (200 train + 100 test images), resize to 128x128, normalise to [0, 1]. The 200 train images are split 160/40 into train/validation; the 100 official test images are the test set.

In [ ]:
URL = "https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/bsds/BSDS300-images.tgz"

def download_bsds(root="data"):
    """Download and extract BSDS300 images if not already present."""
    os.makedirs(root, exist_ok=True)
    if not os.path.isdir(os.path.join(root, "BSDS300")):
        tgz = os.path.join(root, "bsds.tgz")
        urllib.request.urlretrieve(URL, tgz)
        with tarfile.open(tgz) as t:
            t.extractall(root)
    return os.path.join(root, "BSDS300", "images")

def load_images(folder):
    """Load all jpgs in a folder as a float32 tensor (N,3,H,W) scaled to [0,1]."""
    files = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    arr = [np.asarray(Image.open(f).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC), dtype=np.float32) / 255.0 for f in files]
    return torch.from_numpy(np.stack(arr)).permute(0, 3, 1, 2).contiguous()

img_root = download_bsds()
train_all = load_images(os.path.join(img_root, "train"))
test_imgs = load_images(os.path.join(img_root, "test"))
perm = torch.randperm(len(train_all), generator=torch.Generator().manual_seed(SEED))
train_imgs, val_imgs = train_all[perm[:160]], train_all[perm[160:]]
print("train/val/test:", train_imgs.shape, val_imgs.shape, test_imgs.shape)

## 2. Noise injection
Two corruptions: additive Gaussian noise (std `sigma`) and salt-and-pepper noise (fraction `amount` of pixels set to 0 or 1). Noise is regenerated every batch, which acts as data augmentation.

In [ ]:
def add_gaussian_noise(x, sigma):
    """Add zero-mean Gaussian noise with std sigma and clip to [0,1]."""
    return (x + sigma * torch.randn_like(x)).clamp(0, 1)

def add_salt_pepper_noise(x, amount):
    """Set a fraction `amount` of pixel locations to 0 (pepper) or 1 (salt), equal probability."""
    r = torch.rand(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device)
    out = x.clone()
    out = torch.where(r < amount / 2, torch.zeros_like(out), out)
    out = torch.where((r >= amount / 2) & (r < amount), torch.ones_like(out), out)
    return out

def corrupt(x, kind, level):
    """Dispatch to the requested noise type."""
    return add_gaussian_noise(x, level) if kind == "gaussian" else add_salt_pepper_noise(x, level)

# Training-time default levels
DEFAULT_LEVEL = {"gaussian": 0.1, "saltpepper": 0.1}

## 3. Model architecture
Convolutional DAE for 3x128x128 input.

**Encoder** (4 stride-2 3x3 convs, BatchNorm + ReLU): 3 -> 32 -> 64 -> 128 -> 256 channels, spatial 128 -> 8. A final 1x1 conv maps to `bottleneck_ch` channels, giving a latent of `bottleneck_ch x 8 x 8` values (bottleneck size = 64 * bottleneck_ch; default 16 channels = 1024 values, 4.2% of the 49,152 input values).

**Decoder** mirrors the encoder with 4 transposed convs (4x4, stride 2), BatchNorm + ReLU, and a final **Sigmoid** to output pixels in [0, 1].

**Loss:** MSE between the reconstruction and the clean image.

In [ ]:
class DenoisingAutoencoder(nn.Module):
    """Convolutional denoising autoencoder with a configurable bottleneck channel count."""
    def __init__(self, bottleneck_ch=16):
        super().__init__()
        def down(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        def up(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.encoder = nn.Sequential(down(3, 32), down(32, 64), down(64, 128), down(128, 256),
                                     nn.Conv2d(256, bottleneck_ch, 1))           # latent: bottleneck_ch x 8 x 8
        self.decoder = nn.Sequential(nn.Conv2d(bottleneck_ch, 256, 1), nn.ReLU(True),
                                     up(256, 128), up(128, 64), up(64, 32),
                                     nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))

m = DenoisingAutoencoder().to(DEVICE)
print("params:", sum(p.numel() for p in m.parameters()), "| latent shape:", m.encoder(torch.zeros(1, 3, 128, 128, device=DEVICE)).shape)

## 4. Training
Adam (lr 1e-3, halved on validation plateau), MSE loss. Each batch is corrupted on the fly with the chosen noise type. The best validation checkpoint is kept.

In [ ]:
def make_loader(imgs, bs, shuffle):
    """Wrap an image tensor in a DataLoader (noise is added later, on the GPU)."""
    return DataLoader(imgs, batch_size=bs, shuffle=shuffle)

@torch.no_grad()
def evaluate_loss(model, imgs, kind, level):
    """Mean MSE of the model on a set of images corrupted with the given noise."""
    model.eval(); tot = 0
    for x in make_loader(imgs, 32, False):
        x = x.to(DEVICE); tot += nn.functional.mse_loss(model(corrupt(x, kind, level)), x, reduction="sum").item()
    return tot / imgs.numel()

def train_dae(kind, level, bottleneck_ch=16, epochs=60, lr=1e-3, bs=16, verbose=True):
    """Train a DAE on noisy->clean pairs and return (best model, history dict)."""
    model = DenoisingAutoencoder(bottleneck_ch).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5)
    best, best_state, hist = 1e9, None, {"train": [], "val": []}
    for ep in range(epochs):
        model.train(); tot = 0
        for x in make_loader(train_imgs, bs, True):
            x = x.to(DEVICE)
            loss = nn.functional.mse_loss(model(corrupt(x, kind, level)), x)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(x)
        tr, va = tot / len(train_imgs), evaluate_loss(model, val_imgs, kind, level)
        sched.step(va); hist["train"].append(tr); hist["val"].append(va)
        if va < best: best, best_state = va, {k: v.clone() for k, v in model.state_dict().items()}
        if verbose and (ep % 10 == 0 or ep == epochs - 1): print(f"[{kind} {level} b{bottleneck_ch}] ep {ep+1:3d} train {tr:.5f} val {va:.5f}")
    model.load_state_dict(best_state)
    return model, hist

model_g, hist_g = train_dae("gaussian", 0.1)
model_sp, hist_sp = train_dae("saltpepper", 0.1)
for h, n in [(hist_g, "Gaussian"), (hist_sp, "Salt&Pepper")]:
    plt.plot(h["train"], label=f"{n} train"); plt.plot(h["val"], "--", label=f"{n} val")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Training curves"); plt.show()

## 5. Evaluation and visualisation
Metrics on the test set: MSE, PSNR (dB) and SSIM, compared against the noisy input as a baseline.

In [ ]:
@torch.no_grad()
def test_metrics(model, kind, level):
    """Return dict of mean MSE / PSNR / SSIM for denoised output and for the noisy input (baseline)."""
    torch.manual_seed(0); model.eval(); res = {k: [] for k in ["mse", "psnr", "ssim", "noisy_psnr", "noisy_ssim"]}
    for x in make_loader(test_imgs, 25, False):
        x = x.to(DEVICE); n = corrupt(x, kind, level); y = model(n)
        mse = ((y - x) ** 2).mean(dim=(1, 2, 3))
        res["mse"] += mse.tolist(); res["psnr"] += (10 * torch.log10(1 / mse)).tolist()
        nm = ((n - x) ** 2).mean(dim=(1, 2, 3)); res["noisy_psnr"] += (10 * torch.log10(1 / nm)).tolist()
        for a, b, c in zip(x.cpu().numpy(), y.cpu().numpy(), n.cpu().numpy()):
            res["ssim"].append(ssim_fn(a, b, channel_axis=0, data_range=1.0))
            res["noisy_ssim"].append(ssim_fn(a, c, channel_axis=0, data_range=1.0))
    return {k: float(np.mean(v)) for k, v in res.items()}

@torch.no_grad()
def show_results(model, kind, level, n=5):
    """Plot clean / noisy / denoised triplets for the first n test images."""
    torch.manual_seed(1); model.eval()
    x = test_imgs[:n].to(DEVICE); noisy = corrupt(x, kind, level); y = model(noisy)
    fig, ax = plt.subplots(3, n, figsize=(3 * n, 9))
    for i in range(n):
        for r, (img, t) in enumerate([(x, "Original"), (noisy, "Noisy"), (y, "Denoised")]):
            ax[r, i].imshow(img[i].permute(1, 2, 0).cpu().clamp(0, 1)); ax[r, i].axis("off")
            if i == 0: ax[r, i].set_title(t, loc="left")
    plt.suptitle(f"{kind} noise, level {level}"); plt.tight_layout(); plt.show()

for model, kind in [(model_g, "gaussian"), (model_sp, "saltpepper")]:
    print(kind, test_metrics(model, kind, DEFAULT_LEVEL[kind]))
    show_results(model, kind, DEFAULT_LEVEL[kind])

## 6. Experimental study
**(a) Noise level:** train/test a separate model per noise level (Gaussian sigma and salt-and-pepper amount).  
**(b) Bottleneck size:** vary the bottleneck channels (latent size = 64 x channels) at fixed noise 0.1. Epochs are reduced to 30 per run to keep the study affordable; raise for better numbers.

In [ ]:
import pandas as pd
EXP_EPOCHS = 30

rows = []
for kind, levels in [("gaussian", [0.05, 0.1, 0.2, 0.3]), ("saltpepper", [0.05, 0.1, 0.2, 0.3])]:
    for lv in levels:
        mdl, _ = train_dae(kind, lv, 16, EXP_EPOCHS, verbose=False)
        rows.append({"noise": kind, "level": lv, **test_metrics(mdl, kind, lv)})
df_noise = pd.DataFrame(rows).round(4); print(df_noise.to_string(index=False))

rows = []
for ch in [2, 4, 8, 16, 32, 64]:
    for kind in ["gaussian", "saltpepper"]:
        mdl, _ = train_dae(kind, 0.1, ch, EXP_EPOCHS, verbose=False)
        rows.append({"noise": kind, "bottleneck_ch": ch, "latent_size": ch * 64, **test_metrics(mdl, kind, 0.1)})
df_bn = pd.DataFrame(rows).round(4); print(df_bn.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for kind in ["gaussian", "saltpepper"]:
    d = df_noise[df_noise.noise == kind]
    ax[0].plot(d.level, d.psnr, "o-", label=f"{kind} denoised"); ax[0].plot(d.level, d.noisy_psnr, "x--", label=f"{kind} noisy input")
    d = df_bn[df_bn.noise == kind]; ax[1].plot(d.latent_size, d.psnr, "o-", label=kind)
ax[0].set_xlabel("noise level"); ax[0].set_ylabel("PSNR (dB)"); ax[0].set_title("PSNR vs noise level"); ax[0].legend()
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("latent size (values)"); ax[1].set_ylabel("PSNR (dB)"); ax[1].set_title("PSNR vs bottleneck size"); ax[1].legend()
plt.tight_layout(); plt.savefig("q1_experiments.png", dpi=150); plt.show()
df_noise.to_csv("q1_noise_results.csv", index=False); df_bn.to_csv("q1_bottleneck_results.csv", index=False)

## Discussion (fill in after running)
- **Noise level:** PSNR/SSIM should fall as noise grows; compare the gain over the noisy baseline for Gaussian vs salt-and-pepper (impulse noise is usually harder because corrupted pixels carry no information about the original).
- **Bottleneck size:** very small latents blur fine detail (lower PSNR/SSIM); gains saturate once the latent is large enough, after which the model approaches an identity mapping with less denoising pressure.